<a href="https://colab.research.google.com/github/frankheine/Voice_Gen/blob/main/Voice_Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y sox libsox-fmt-all ffmpeg
!pip install qwen-tts soundfile pysoundfile librosa
!ffmpeg -y -i sample.m4a sample_ref.wav 2>/dev/null && echo "Converted sample.m4a → sample_ref.wav"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3 libsox-fmt-oss
  libsox-fmt-pulse libsox3 libwavpack1
Suggested packages:
  libaudio2 libsndio6.1
The following NEW packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-all libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3
  libsox-fmt-oss libsox-fmt-pulse libsox3 libwavpack1 sox
0 upgraded, 16 newly installed, 0 to remove and 67 not upgraded.
Need to get 800 kB of archives.
After this operation, 2,533 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libao-common all 1.2.2+20180113-1.1ubuntu3 [6,568 B]
Get:2 ht

In [ ]:
# ==============================================================================
# 1. SETUP & INSTALLATION (Run this once per session)
# ==============================================================================
# Run these in a Colab cell first with a ! before them:
# !apt-get install -y sox libsox-fmt-all ffmpeg
# !pip install qwen-tts soundfile

import os
import re
import gc
import torch
import soundfile as sf
from google.colab import drive
from qwen_tts import Qwen3TTSModel
import subprocess


********
********
 


In [ ]:
# ==============================================================================
# 2. MOUNT DRIVE & CONFIGURE PATHS
# ==============================================================================
drive.mount('/content/drive')

# Create a dedicated project folder on your Google Drive
PROJECT_NAME = "Jamie TedTalk"
OUTPUT_DIR = f"/content/drive/MyDrive/Audiobook_Outputs/{PROJECT_NAME}"
CHUNKS_DIR = f"{OUTPUT_DIR}/chunks"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

# ==============================================================================
# 3. INITIALIZE MODEL (Using SDPA for T4 GPU compatibility)
# ==============================================================================
print("Loading model into GPU...")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.float16, # Changed from bfloat16 to float16 to reduce VRAM usage
    attn_implementation="sdpa" # Bypasses the flash-attn compilation freeze
)

# ==============================================================================
# 4. VOICE CLONING PROFILE
# ==============================================================================
# Ensure this file is uploaded to your Colab session folder (not Drive, or update path)
ref_audio_path = "sample_ref.wav"
ref_transcript = "You're not required to track anybody's identity. So my, that's where, so it shows that they paid strike. That's how I'm going to do it for my for the subscription service."

print("Generating voice profile...")
voice_prompt = model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=ref_transcript,
    x_vector_only_mode=False
)

# ==============================================================================
# 5. SMART TEXT CHUNKING (Prevents OOM and handles abbreviations)
# ==============================================================================
raw_text = """
[text here]And then it would be this crybaby victim, “I’m so hurt now” when I eventually got sick and tired of being beat with an inch of my life and then she’s gonna pull out her phone and she’s calling her grandpa and her uncle who by the way her uncle is the vice president of the largest bridge contracting company in Central Illinois who I worked for and is a scary motherfucker so is her grandfather if he wasn’t so old, but I’ve seen those two sit and laugh while their wives tell the story of when they were at the Local Pekin diner 24 hour degenerate hang out which is like the mom and Pop family owned version of a Denny’s and this town used to be filled with bars and live music back in the 80s and 70s. They grew up with the guys that went onto form the band Mudvayne, they are all from Pekin with the same hateful evil genetics as everybody else in Pekin.

Pekin was a major KKK hub back in the 1920s and the KKK owned the Pekin newspaper and would print white supremacist propaganda in it during the time that they owned it 1922 through 1925, my house was built in 1922. I could walk to the Pekin Time building from my house in probably 10 minutes or less. I live right near the historic downtown area and the KKK used the second floor of that building as their clubhouse. Her grandfather told me he’s been to a KKK rally before he goes. I heard there was a kegger going on right over where the present day Pekin Walmart is, which wasn’t there at the time. I rode my bike back there and when I got there, I realized it was a KKK rally and they were burning crosses and had the outfits with the white hoods covering their faces and everything just all people’s parents and stuff it’s a town present day of only 40,000 people.

Pekin was what was referred to as a sundown which means that it was a city ordinance that if any Black people were within the city limits of Pekin after dark, they would be shot on site. I can drive 15 minutes and be in the ghetto where there’s nothing but Black people, but the Illinois which I live just on the other side of acted as a geographic boundary, and they used to be a sign on the other side of the river, which is only about five minutes from my house that said no blacks aloud after 9 PM. Don’t let the sun go down on your black ass… something like that. I’ve tried to dive into the historical archives and have not successfully found a picture of the sign, but I heard that it might’ve been up all the way into the 2000 or 1990s.

I have black friends who live in Peoria probably 20 minutes away from me or so and I’ve gotten some of them to come over to my house before it took a lot of convincing, but there are still several people to this day that go, “Frank, I love you. I realize that your house that you own is in Pekin, but I’m not crossing that river. I’m sorry I’m not coming over to your house.” Their mothers instilled this in their head so stringently telling them whatever you do do not cross that river and go over there to Pekin. They’ll fucking kill you.” so the grandchildren of all the evil KKK people murdering Black people over here are the ones who live here now still and they even look funny around here because I think they’re in bread because they all just stayed in Pekin and they act like they’re retarded too, and even the hottest chick that you could is maybe a Chicago six at best… from the inbreeding, I believe because they got some ugly ass hoes around here. You walk through the Walmart and oh my gosh it’s like the hills have eyes like there’s just a bunch of mutants walking around in there or something, gross.

So to give you an idea of the type of people that my ex-wife’s uncle and grandfather were, whose DNA traces right back to Pekin, which I believe had an influx of people from the south who settled here to work at caterpillar because that’s the only reason Peoria even exists. I only live 2 1/2 hours south of Chicago but holy fuck you would think that you’re in Alabama or Mississippi or some shit based on the way people talk and they have confederate flags mounted on their pickup trucks, we’re not talking about a confederate flag bumper sticker on their back windshield. There’s plenty of those too, but I’m talking literal full-size flags on a stick and they’ll have two of them bitches. One mounted at each tail light from inside the flatbed. I even have a hillbilly truck too. That’s all rusty and shitty like that which I use when I gotta move some shit. All these dip shits that buy these $90,000 dual axle pick up trucks with four wheels in the back and all that shit and then they don’t even haul anything in it at all other than their fat girlfriend or wife it’s just part of the culture that they think that demonstrates their manliness because of how big their truck is and they love to put lift kits on them too, and they put stickers that say salt life because they’re in the fishing, but I mean last time I checked, I’m no fishing expert, but we’re nowhere near notion so there’s no fucking saltwater in there, but like I said, the people around here are fucking retarded. They also love to put stickers of a cartoon character like he’s peeing on various different phrases or they like to have it where he’s peeing on the name “BIDEN”, and middle finger stickers are also very popular or anything that explicitly uses the word fuck that’s displayed for everyone to see because that makes them extra cool… what a bunch of slackjawed faggots..

I’ve heard her grandfather and Uncle, the one who was the vice president of the construction company I worked for they would laugh and giggle as their wives would do the same while fondly telling the story about how when they were at that diner one night some guy said something about one of their wives who they were dating at the time this must’ve been back in the 1980s or 90s maybe and they worked as carpenters even back then. So they did the whole, “let’s take this outside” and proceeded to get a 4 foot wooden level and jumper cables out of thel of one of their pickup trucks, and then proceeded to beat the absolute living fuck out of this guy, and whoever he was with by using the jumper, cables and the level as weapons. And the whole family would all join in laughter.

So right after I’d finally have to do something drastic before my wife literally fucking killed me during one of these episodes where she was just teeing off on me and I eventually would grab hold of her and get her off of me and do whatever I have to while I’m fighting for my life somebody that literally almost weighs twice as much as Me. I think she got as fat as 240 pounds at one point like a fat evil demon, she, I can actually confirm is in fact, inbred, and not even because of Pekin, but through her other trash lineage on the other side of her family, which she was very ashamed of and told me this only in secret saying that her grandparents are great grandparents were cousins or some shit like that so it’s no wonder they’re all a bunch of fucking dipshits. So she would do the whole, “ you hit me. I’m calling my grandpa and Kevin and I’m gonna tell them what you did and they’re gonna come over here with guns and they’re gonna fucking kill you.” so then I’d have to wrestle with her to take her cell phone away from her to keep her from calling these fucking animals and telling them lies that could put my safety or life in danger.

Her entire family are all such losers that there’s not a single person in her entire family who has ever graduated high school. When I had finally after years of convincing and pressuring my wife to at least get her GED have some kind of level of ability that you get a job and not have to ride the short bus in order to get there with the rest of the retards. You would’ve fought she graduated Harvard with a PhD because the whole family was that proud of her for getting a fucking GED so that hopefully paints the picture of just how big of lowlife scum these disgusting farm animals are.

I realized that what makes me happiest in life is for me to come up with ways to figure out how to make enough money to support my lifestyle without having to have a regular job or some fucking dickhead telling me what to do so that I can free up my time to be an artist and create l and write things. If I were to get a regular job, I would never be able to reach my full potential. The only way for me to do so is to be crazy enough to risk it all and go all in and follow what I know to be the correct path because I believe God is showing me the way and putting me through the struggles and hardships necessary in order to forge me into the greatest version of myself that I will need to be in order to be ready so that I don’t fumble the next level of success that he passes my way and I know it’s coming.

Lots of people say they believe in God, but I think most of them are just lying. I think they were raised and told that they must believe in God and that they think that there’s may be a chance that God exists, but they are unsure, and they don’t know what they believe. They haven’t even read the Bible enough to be able to say whether or not they believe it’s true. That was the case for me, I believe it’s 90% fiction for the New Testament and nearly 100% fiction for the Old Testament, but that’s not where the Bible derives its value from. I believe the Bible is a book of universal truth and timeless, wisdom, cautionary tales warning us, and teaching us about the human condition and the way people will stab you in the back and betray you just like they did to Jesus. I believe Jesus was a real person. I don’t believe he was the Messiah.

But according to the Bible, if you cannot, I’m honestly say wholeheartedly that you don’t have an ounce of doubt that Jesus couldn’t have just been some lunatic claiming to be God, son or that there’s not even the slightest chance that they believe that maybe the whole thing could all be made up and not true that the Bible states, regardless of your behavior and immorality of how you conduct yourself ethically during your lifetime that you’re gonna burn in hell if you don’t wholeheartedly accept Jesus Christ as your personal Lord and Savior… I struggled with this for a long time and realized no of course it’s not true but that’s what they had to do back then in order to get everybody after it was crazy savage times even as it’s depicted in the Old Testament it was anarchy. They have to do something to get everybody to stop murdering raping pillaging stealing everybody’s shit beating everybody up because the human condition is rooted and selfishness. That’s the whole reason we stay alive is because we have that instinct, but when to their own devices, people are evil and do crazy things to serve themselves and that’s why for the masses they have to create a way to have population control so that they could make the world a better place and it worked.

But I swore off religion and God a long time ago because I’m made the mistake of grouping it all together what I realize is that organized religion is man-made and since man is self-serving, of course all religion is tainted and imperfect, and is fake because along the way, selfish people, pretending to be believers in God and pretending to be doing things in the best interest of the greater good for the society we’re really just stacking the deck in their favor.

But I grouped God in with all of this and wrote it off as thinking God‘s not real either that it’s all fake but I promise you experienced it and felt God come and take over my body two different times and He is definitely real and He left me with the sense of, “don’t be scared you’re not out here alone. This isn’t random. Everything happens for a reason. I’m always watching. I’m here with you. I’m not going to let you down. I’m not gonna throw any challenges your way that you are not prepared to handle no matter how terrifying it may seem at the moment, and how impossible it may seem in the moment to surmount that adversity, you’re always gonna overcome it and I’m telling you the sense of fear evaporated from me.

I only started to search to find God when I was at the end of my marriage falling apart. My wife told everybody that I’m an abusive woman beat her and beat her up and turned them all against me and I went from being everybody’s favorite in law to having the rug pulled from under me and my entire Support system vanished overnight. When I have to write down an emergency contact sometimes I just draw a line or write none, it’s terrifying. I had never lived by myself ever I never learned how to be by myself in peace before. I always had to have a TV on music podcast. Something to make it feel like that. Was someone else here with me or some other type of life occurring in my vicinity for me not to feel lonely and then I finally found God and realized this is what I have been missing all along. This is what the rule of the hippie movement of the 60s this is what all these lost souls or taking all these drugs and alcohol for or being overly promiscuous all of these things where it becomes a vice that is being used to fill this void that we have that we can’t figure out why and we don’t really know what we’re even looking for. The reason people take psychedelics and have this feeling that they are in search of something some greater meaning or underlying concept they’ve overlooked that between the lines and that’s what it is. It’s a connection with God that they are searching for, and I found it, all those feelings of being alone, and not being at ease and constantly having this compulsion to consume as much drugs and alcohol as I possibly could, evaporated.

It’s like I’ve become a monk or something, I will be working on something and putting all my focus and energy into it for hours on end in absolute complete total silence with no urge at any point to turn on music or a TV or call somebody for comfort or try to pick up some chick out at a bar somewhere to bring her home with me because deep down really it’s just that I’m lonely… all of that vanished and so did the compulsion to want to constantly be so consumed with all these vices.

Now, having said that, yes, I still use drugs. Has it become a problem while I’m in the 11th hour of total abandonment every social safety net that’s supposed to catch me failed me every system that was supposed to be there as a fallback turned out to be riddled with corruption, and in many cases was actually Weaponized against me and even through all of this that is when I gained this clarity and once I have to either go on probation and have to piss in a cup or go to prison one way or another I’m gonna have to stop and it’s not gonna be hard like before. I’ve been an alcoholic since I was released from prison. There’s only been maybe a day or so here or there that I have gone with no alcohol consumption in nearly a decade. I’ve had the forcibly dry out when I got put in jail. There were times that my wife would quit drinking for a short period of time and I had to do so under her control because she forced me and the longest period Was maybe like a week or two but the compulsion never went away. I was always thinking about it and couldn’t wait for the opportunity so that I could consume as much as possible. But once I got that connection with God where it’s like as if he’s with me all the time seeing all the cool shit I’m doing where I’m completing it and going look at how bad ass that is and I don’t even realize I’m doing this, but it’s as if I feel like there is another person here with me that I’m showing something too and I’m like feeling proud of myself as they look at it and then I realize that I am in fact all alone and have been this whole time, but have this on mistakable sense that somebody is proud as they see the amazing feet I’ve overcame or beautiful creation that I have synthesized and it has to be God there’s no other explanation. I still do have the urge to wanna show it to other people so they can see it but ultimately I’ve found peace in my solidarity and have taken route in it and flourished like never before, and it has become as if it’s my superpower, the fact that I don’t need to have somebody else here in order to engage with or feel like they are keeping me company just to hide from that terrible feeling deep inside of me that I could never cure no matter how much drugs or alcohol I tried to do so with and I think that’s what most people struggle within one way or another. It’s because deep down in their soul they are scared. They are feeling alone and as if everything in the world is just up to chance, because based on the hardcoded rules of our reality that we exist and there’s no possible way that you could have freedom of choice with the seeming ability to have complete autonomous free will, but then also for God to have a predetermined plan for you to follow. That’s already been written those two concepts completely negate the other, they are the absolute antithesis of one another. But all these people walk around, acting like they believe in God, but deep down in their soul they don’t believe that and that’s why they have that impending sense of doom and terrible pain deep inside their soul because they’re odds with themselves. Big wide so hard to themselves for so long that they’ve begun to believe their own lies, but ultimately what this comes down to is regardless if somebody is addicted to drugs and alcohol or not, it’s the human condition of selfish hubris which they worship as their false deity that they worship and that’s why they have no relationship with God because it’s not possible when all they care about is fulfilling their own selfish needs and pretending to care about others, but ultimately not caring about anyone but themselves. Look at my mother she stands up in the front of the church and she is just putting on the whole production of how holy she is in front of everyone. I think she even believes this charade herself but deep down there’s no possible way she believes in God or she be scared to death that he would punish her for the way that she has treated me, but it’s because she’s so blinded by her own selfishness that she’s able to actually succumb to her own lies and really is serving her own selfishness by putting on this act because I believe she thinks that will cause other people to view her in a better life and that they will think she’s a good person and if they think she’s a good person, then that will allow her to manipulate them into doing things that will serve selfishness and it all stems from that same worship of herself and this is why that is such a recurring theme in the Bible and it is absolutely true because it’s a double edged sword if we weren’t selfish, we would never have survived. We would never have been able to have the instinct to perpetuated humanity in this long, but that’s why everybody has this void that they’re constantly trying to fill with all the other things that they worship other than God because even though they say they believe in God in their soul, they know that that’s not true and that they are lying and because they are unsure and who wouldn’t be because look at all the man-made religions that they come up with to confuse everybody and they all lost side of the fact that there’s just one God that created everything and that the Christians, the Jews, the Muslims, the Mormons, the Scientologists, and me are all praying the same God. It’s not by accident that tribes which respond all over the globe and none of them had any ability to talk to one another, but every single one of them always has an obsession with trying to determine or fabricate their origin story of where we came from and some type of entity to worship as a higher power this is built into our DNA. It’s hardcoded right into our being every last one of us on the planet questions that and prioritize that as being a philosophical conundrum that must be solved so they create these religions and none of them are correct.

I mean, it doesn’t get any more obvious than with Catholicism. They preach the 10 Commandments as the rules of life, morality laws to live by, I believe it’s the very first Commandment, the most important one that says essentially, I am your one and only God you worship me and that’s it, you don’t worship the dollar, you don’t worship what your dick tells you to do, you don’t worship food and engage in gluttony and overindulgence, you don’t worship kings or other human beings, you don’t worship statues, you don’t worship psychoactive substances, you worship me and that’s it, and that’s non-negotiable and that’s why I’m putting it as the first one on the list. You guys are all equal you guys are all my children not one of you is better than any of the others you guys are all equal. Catholics literally have an emperor of the Vatican that gets selected by a means of a popularity contest that’s not even of the people, but just some other human beings who are designated to be superior in their opinion of who should become the pope and the pope is essentially treated by the Catholics as though he is some kind of supreme being who has God‘s direct phone number and can talk to him and relay messages back from him and all this shit, meanwhile, the vatican is literally its own sovereign nation that dictates its own laws and is not part of any country. So the fundamentals of the Catholic religion are based around the fact that they literally pick one human being to designate as the supreme superior being who is somehow not equal to the rest of us, and then there are other people who vote the supreme being into power, whose opinions are more valuable than the rest of ours, and they literally all go and sit in some structure and discuss it over and all the people wait and watch the color of the smoke coming out of the chimney and when the smoke changes color, everybody cheers because this is how they let them know that they’ve reached a decision as to who’s gonna be the new pope. And then when all this transgender stuff is going on and all the gay stuff and all this, my philosophy is that what somebody else is doing is not impacting me or the other people around me in a way that is interfering with my own pursuit of liberty who am I to interfere with their decisions? I don’t think it’s wrong to be gay or transgender. Do I think these transgender people are all just confused and due to being validated by their peers because really deep down they’re just gay and confused and don’t feel comfortable in their own skin like every other teenager that’s ever lived and then society goes good job. We’re so proud of you. You are so brave for coming out in transitioning like that and people start seeing that and then they go you know what I think I am transvestite that’s why I feel so uncomfortable in my skin. I don’t think there’s anything morally wrong where God is gonna smite them for that or punish them, but that’s what the Bible says. The Bible also depicts the use of slavery. So it became unpopular and inconvenient to continue perpetuating what the Bible has always said about gay people, the pope literally said that they’re changing the rules and they’re gonna let the gay people become part of the Catholic religion now and that God‘s cool with it now even though he wasn’t before, but the pope says so, and he is apparently the way the Catholics treat him as though he’s some figure that resembles a messiah like Jesus but it’s literally just some guy that they voted for and is done so by an organization that has literally covered up child predators going around raping kids, and not just covering it up, but not notifying anybody about it and then taking the same child monster and putting them into new community somewhere else in the world giving him a fresh start with they can repeat the cycle of abuse again to the next unsuspecting community that they are planted in.

So it’s it pretty harsh example with the Catholics, but that is the truth of it and that’s what put a bitter taste in my mouth and made me swear off God altogether, but since I’ve actually wholeheartedly concluded that God is real and is connected with me in a way that I have an actual relationship with God, which I always thought sounded so silly, but it’s like when I began soul, searching and getting to the route of my psyche, and began sort of freeing myself from fixation on worldly possessions, and all the shiny fancy things that I chased because I thought that would fill that void and never did and this is what I believe goes on with the majority of people in our country anyhow, is that they’re always after the shiny (New) thing. They need something to Chase because they think that’s gonna be the thing that makes them happy and it never does. It’s never enough just like when you’re trying to fill the void with drugs or alcohol or sex or thrillseeking with reckless behavior or gambling pissing away every dollar they earned in the slot machine, that’s what the people around here do they love slot machines, they’re too stupid to realize they’re just literally throwing their money in the garbage. Why don’t you just go spend it on some fucking drugs? That’s what I try to tell him cause they of course are all addicted to drug us to everybody that does math loves doing slot machines too it’s like it brings them some kind of a high that thank goodness doesn’t happen to me or I would be completely broke. Same thing for smoking crack just doesn’t do the same thing to me as everybody else thank goodness because I know some crackheads and it is sad to watch how it fucks their mind up to where I don’t even think they’re getting high. I’m convinced maybe they got high like once when they first smoked it and that they’ve been chasing that ever since and it’s like they spend their whole day going on all these quests and trying to get their next person, they can run a scheme on to get some and then fast as I can do what they gotta go to the ghetto and get it from the crack dealer and then they go, “yep, we will go hang out and do this activity in just a minute here. I’ve got everything I need now, I’m just gonna smoke one up real quick and then we’ll get to it.” and they just keep smoking it and smoking it until it’s gone and never seem to get satisfied from it for even a moment and then when it’s gone, they have to repeat the cycle and figure out what scheme they’re gonna pull in order to go get more money to go back it’s just literally consumes every bit of their time and I can just see the way they are making themselves miserable, but are so blinded by their own selfish hubris that they don’t even realize what is happening, just like when you watch them play the slot machines. I will ask them individually one on one if they believe over their lifetime of playing slot machines that they are ahead behind or broken, even and every last one of them all with total confidence will try to convince me that they have at least broken even but might actually be Ahead even though I watch them repeatedly just dump all their money into these machines, their brain must delete all the times that they lost and just remembers the times that they won or if there’s someone else that they heard about that one, a couple thousand dollars recently it’s like that becomes the talk of the town and is what perpetuates everybody else’s gambling addiction because they go all well if Tammy-Lynn one $800 last week, then I’m overdue and I’ve got my lucky rabbits foot and my lucky lottery ticket scratcher with me and have my lucky underwear on so I think today is gonna be the day. Sometimes like Call them out on it and tell them how fucking dumb they sound and try to get them to understand that the lottery and slot machines is just a literal tax on poor people and retards. I will ask them too, have you ever seen a rich person ever put a single fucking diamond one of those things? I can tell you right now around lots of rich people and not one of them would ever in 1 million years be so foolish as the stick a single fucking penny into one of those things.

To be continued…

"""

def smart_chunk_text(text, max_chars=450):
    # 1. Replace smart typography FIRST
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2014", " - ").replace("\u2026", "...")
    # 2. Now strip remaining non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # 3. Group full sentences together up to max_chars
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current) + len(sentence) + 1 <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            while len(sentence) > max_chars:
                split_point = sentence[:max_chars].rfind(' ')
                if split_point == -1:
                    split_point = max_chars
                chunks.append(sentence[:split_point].strip())
                sentence = sentence[split_point:].strip()
            current = sentence

    if current:
        chunks.append(current)
    return chunks


text_chunks = smart_chunk_text(raw_text)
print(f"Total chunks to process: {len(text_chunks)}")

# ==============================================================================
# 6. GENERATION LOOP (With Resume Capability & VRAM Management)
# ==============================================================================
failed_chunks = []
for idx, chunk in enumerate(text_chunks):
    chunk_filename = os.path.join(CHUNKS_DIR, f"chunk_{idx:04d}.wav")
    # Resume capability: Skip if chunk already exists
    if os.path.exists(chunk_filename):
        print(f"  Chunk {idx} already exists, skipping.")
        continue

    print(f"Generating chunk {idx+1}/{len(text_chunks)}...")

    try:
        with torch.no_grad():
            wavs, sample_rate = model.generate_voice_clone(
                text=[chunk],
                language=["English"],
                voice_clone_prompt=voice_prompt
            )
        sf.write(chunk_filename, wavs[0], sample_rate)
        del wavs
    except Exception as e:
        print(f"  ⚠️ Chunk {idx} failed: {e} — skipping")
        failed_chunks.append((idx, chunk, str(e)))
    finally:
        gc.collect()
        torch.cuda.empty_cache()


# ==============================================================================
# 7. ZERO-RAM STITCHING VIA FFMPEG
# ==============================================================================
if failed_chunks:
    print(f"\n{len(failed_chunks)} chunks failed:")
    for idx, chunk, err in failed_chunks:
        print(f"  Chunk {idx}: {err}")

print("Stitching chunks into final audiobook...")

# Create a text file listing all chunks in order for ffmpeg
list_file_path = f"{OUTPUT_DIR}/concat_list.txt"
with open(list_file_path, "w") as f:
    # Read files in sorted order to ensure correct sequence
    chunk_files = sorted([f for f in os.listdir(CHUNKS_DIR) if f.endswith('.wav')])
    for chunk_file in chunk_files:
        # ffmpeg requires the format: file 'path/to/file.wav'
        f.write(f"file '{CHUNKS_DIR}/{chunk_file}'\n")

final_output_path = f"{OUTPUT_DIR}/{PROJECT_NAME}_Complete.mp3"

# Run FFmpeg to concatenate without loading audio into memory
ffmpeg_command = [
    "ffmpeg", "-y", "-f", "concat", "-safe", "0",
    "-i", list_file_path,
    "-b:a", "192k", # High quality MP3 bitrate
    final_output_path
]

subprocess.run(ffmpeg_command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(f"🎉 Audiobook successfully created and saved to: {final_output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model into GPU...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
!ffmpeg -y -i sample.m4a sample_ref.wav
!ls -lh sample_ref.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab